# LatentEuclid — VL-JEPA Phase 1: Trainable Y-Encoder (LoRA)

Minimal extension of `training/train_x_encoder.py` — the only architectural change is that
the Y-Encoder is now **trained jointly** (LoRA r=16) instead of using frozen offline `.pt` targets.

**What is unchanged from the original:**
- X-Encoder (`LatentEuclid`): full VLM + predictor MLP training, same LR and loss
- Loss: `huber_cosine` (works at micro-batch size 1-2)
- Checkpoint rotation and augmentation flag (both default to off in config)

**What changes:**
- Y-Encoder: `Qwen/Qwen3-VL-4B-Instruct` with LoRA r=16 instead of frozen Qwen3-0.6B
- Y-Encoder input: image + question + cumulative step text (VLM, not text-only)
- Target embeddings computed live per batch — no pre-built `.pt` files needed
- Step-token masking in `compute_live_targets` mirrors `embed_steps_batch()` in `build_manifold.py`
- **Image source**: Phase 1 loads images from HuggingFace by index; `train_x_encoder.py` loads from disk paths in the JSONL. Same images, different I/O path.
- **Optimizer**: two param groups (`LR_X` for X-Encoder, `LR_Y` for Y-LoRA); original uses a single group.

**Checkpoint**: predictor + LoRA adapters only (~200 MB). X-VLM weights recovered from HuggingFace.

## Cell 1 — Environment setup

In [1]:
import sys, os

REPO_PATH      = "/home/ec2-user/work/MMML-Project"
CHECKPOINT_DIR = "/home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora"
DATA_CACHE     = "/home/ec2-user/work/MMML-Project/data/geothought_hf"

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Repo:        {REPO_PATH}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

Repo:        /home/ec2-user/work/MMML-Project
Checkpoints: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora


In [2]:
import torch
assert torch.cuda.is_available(), "GPU not found — check CUDA installation"
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

GPU:  NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


## Cell 2 — Dataset download

Images from HuggingFace · Reasoning steps from `geothoughts_verified.jsonl`.
Run once per session; reloads from local cache on subsequent runs.

In [3]:
from datasets import load_dataset, load_from_disk

if os.path.exists(DATA_CACHE):
    print("Loading HF dataset from local cache...")
    hf_dataset = load_from_disk(DATA_CACHE)
else:
    print("Downloading from HuggingFace (one-time, will cache to disk)...")
    hf_dataset = load_dataset("xinlingdedeng/Geo-Thought", split="train")
    hf_dataset.save_to_disk(DATA_CACHE)
    print(f"Saved to {DATA_CACHE}")

print(f"Dataset size: {len(hf_dataset)} samples")

Loading HF dataset from local cache...
Dataset size: 17077 samples


## Cell 3 — Dataset

`JointTrainDataset` returns image + question + K=4 cumulative reasoning steps.
Structurally mirrors `GeoThoughtsDataset` in `train_x_encoder.py` — same JSONL,
same 90/10 split, same thought-token injection.

**Flagged difference**: images come from HuggingFace by index instead of disk paths in
the JSONL. Same underlying images; different I/O. `train_x_encoder.py` uses `Image.open(img_path)`
(line 79); this dataset uses `hf_dataset[hf_idx]`. No pre-built `.pt` target files required.

In [4]:
import json
import re as _re
from torch.utils.data import Dataset, DataLoader
from data.build_manifold import parse_k4_steps


class JointTrainDataset(Dataset):
    # Image + question + K=4 cumulative reasoning steps per sample.
    # Images sourced from HuggingFace by index; steps from geothoughts_verified.jsonl.
    def __init__(self, jsonl_path, hf_dataset, k_steps=4):
        self.hf_dataset = hf_dataset
        self.k_steps = k_steps
        self.data = []
        with open(jsonl_path) as f:
            for idx, line in enumerate(f):
                item = json.loads(line)
                steps = parse_k4_steps(item.get("reasoning", ""))
                if len(steps) == k_steps:
                    m = _re.search(r'sample_(\d+)', item.get("image_path", ""))
                    hf_idx = int(m.group(1)) if m else idx
                    self.data.append({
                        "hf_idx":    hf_idx,
                        "question":  item["question"],
                        "steps":     steps,
                    })
        print(f"Loaded {len(self.data)} valid samples.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        hf_row = self.hf_dataset[item["hf_idx"]]
        image = hf_row.get("image") or hf_row.get("images")
        if image is None:
            from PIL import Image as _PIL
            image = _PIL.new("RGB", (224, 224), (128, 128, 128))
        elif not hasattr(image, "convert"):
            from PIL import Image as _PIL
            image = _PIL.fromarray(image)
        image = image.convert("RGB")
        thought_string = "".join([f"<thought_{i+1}>" for i in range(self.k_steps)])
        return {
            "image":    image,
            "text":     item["question"] + " " + thought_string,
            "steps":    item["steps"],
            "question": item["question"],
        }


def joint_collate(batch):
    return (
        [b["image"]    for b in batch],
        [b["text"]     for b in batch],
        [b["steps"]    for b in batch],
        [b["question"] for b in batch],
    )


JSONL_PATH = os.path.join(REPO_PATH, "data/geothoughts_verified.jsonl")
full_dataset = JointTrainDataset(JSONL_PATH, hf_dataset)

train_size = int(0.9 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Loaded 5941 valid samples.
Train: 5346 | Val: 595


## Cell 4 — Training configuration

Mirrors `config.yaml` `train_x_encoder` section. The only new hyperparameters are
`LORA_R` and `LR_Y` for the Y-Encoder adapters.

In [5]:
EXPERIMENT_NAME  = "vl_jepa_lora_v1"
BASE_MODEL_ID    = "Qwen/Qwen3-VL-4B-Instruct"   # X-Encoder backbone (unchanged)
TARGET_MODEL_ID  = "Qwen/Qwen3-VL-4B-Instruct"   # Y-Encoder backbone — same family, latent dim 2560
K_STEPS          = 4
LORA_R           = 16                              # Y-Encoder LoRA rank

# Training — match train_x_encoder config values.
# Original: batch_size=16, grad_accum=8  ->  effective batch = 128
# Notebook: micro_batch=8 (RTX PRO 6000 Blackwell, 102 GB), grad_accum=16  ->  effective batch = 128
MICRO_BATCH_SIZE    = 8
GRAD_ACCUM_STEPS    = 16
LR_X                = 1e-4    # X-Encoder — kept at config.yaml value for comparison integrity
LR_Y                = 5e-6    # Y-Encoder LoRA — lower to avoid collapse
WEIGHT_DECAY        = 0.01
MAX_GRAD_NORM       = 10.0    # natural grad norm ~1-10 during stable training; 10 catches spikes without destroying signal
WARMUP_STEPS        = 20      # ~half an epoch; enough to avoid destructive first steps
EARLY_STOP_PATIENCE = 10      # val checks without improvement before halting
EPOCHS              = 100
SAVE_EVERY_N_STEPS  = 20      # also used as val interval (VAL == SAVE)

DEVICE = "cuda"
print(f"Effective batch size: {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS}")

Effective batch size: 128


## Cell 5 — Load models

- **X-Encoder** (`LatentEuclid`): same full training as `train_x_encoder.py`
  — VLM backbone + predictor MLP both trained at `LR_X`.
- **Y-Encoder** (`Qwen3-VL-4B + LoRA`): loaded via `load_qwen_target_model` (reused from
  `build_manifold.py`), then wrapped with LoRA r=16.

Both models use gradient checkpointing to reduce activation memory.

In [6]:
import gc

for _name in ["x_model", "y_encoder", "optimizer", "criterion"]:
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
_free  = torch.cuda.mem_get_info()[0] / 1e9
_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM free: {_free:.1f} / {_total:.1f} GB")

VRAM free: 101.4 / 102.0 GB


In [7]:
from models.latent_euclid import LatentEuclid
from training.stable_alignment_loss import AlignmentLossFactory
from data.build_manifold import load_qwen_target_model
from peft import LoraConfig, get_peft_model

# ── X-Encoder — identical to train_x_encoder.py ──────────────────────────────
print("Loading X-Encoder (LatentEuclid)...")
x_model = LatentEuclid(
    base_model_id=BASE_MODEL_ID,
    target_model_id=TARGET_MODEL_ID,
    max_thought_tokens=K_STEPS,
)
x_model.vlm.gradient_checkpointing_enable()

# ── Y-Encoder — Qwen3-VL-4B + LoRA ───────────────────────────────────────────
print("Loading Y-Encoder base via load_qwen_target_model...")
tokenizer_y, processor_y, y_base = load_qwen_target_model(TARGET_MODEL_ID, DEVICE)

if tokenizer_y.pad_token is None:
    tokenizer_y.pad_token = tokenizer_y.eos_token
tokenizer_y.padding_side = "right"
processor_y.tokenizer = tokenizer_y

print("Applying LoRA to Y-Encoder...")
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
y_encoder = get_peft_model(y_base, lora_cfg)
y_encoder.enable_input_require_grads()
y_encoder.gradient_checkpointing_enable()
y_encoder.print_trainable_parameters()

# ── Loss — same as train_x_encoder.py ────────────────────────────────────────
criterion = AlignmentLossFactory(loss_type="huber_cosine")

# ── Optimizer — X trained fully (same as original) + Y LoRA ──────────────────
optimizer = torch.optim.AdamW([
    {"params": x_model.vlm.parameters(),       "lr": LR_X, "name": "x_vlm"},
    {"params": x_model.predictor.parameters(), "lr": LR_X, "name": "x_predictor"},
    {"params": [p for p in y_encoder.parameters() if p.requires_grad],
                                               "lr": LR_Y, "name": "y_lora"},
], weight_decay=WEIGHT_DECAY)

n_x = sum(p.numel() for p in x_model.parameters()) / 1e6
n_y = sum(p.numel() for p in y_encoder.parameters() if p.requires_grad) / 1e6
print(f"Trainable — X-Encoder: {n_x:.0f} M | Y-LoRA: {n_y:.1f} M")

Loading X-Encoder (LatentEuclid)...


Added 4 dynamically routed new thought tokens


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Base LatentEuclid Vision-Language Model (Qwen/Qwen3-VL-4B-Instruct)...


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Loading Y-Encoder base via load_qwen_target_model...
Loading Qwen/Qwen3-VL-4B-Instruct for Target Manifold extraction on cuda...


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Applying LoRA to Y-Encoder...
trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651
Trainable — X-Encoder: 4450 M | Y-LoRA: 11.8 M


In [8]:
# Set RESUME = True to continue from the latest checkpoint.
RESUME = True

start_epoch = 0
global_step = 0

if RESUME:
    # Step 1: restore the full X-encoder (VLM backbone + predictor) from best.pt.
    # best.pt always contains model_state_dict (saved with full=True), so trained
    # VLM weights are preserved. Also restore y_lora and epoch/step from best.pt
    # as the baseline — step checkpoint overlay (Step 2) will supersede these if
    # a real step checkpoint exists.
    best_path = os.path.join(CHECKPOINT_DIR, "best.pt")
    if os.path.exists(best_path):
        print(f"Loading full checkpoint from {best_path}...")
        best_cp = torch.load(best_path, map_location="cpu", weights_only=False)
        x_model.load_state_dict(best_cp["model_state_dict"])
        # Restore Y-LoRA from best.pt as the baseline.
        # Step 2 will overlay a more recent version if one exists.
        if "y_lora" in best_cp:
            with torch.no_grad():
                for name, param in y_encoder.named_parameters():
                    if name in best_cp["y_lora"]:
                        param.copy_(best_cp["y_lora"][name])
            print("  Restored Y-LoRA from best.pt")
        start_epoch = best_cp["epoch"]
        global_step = best_cp["global_step"]
        print(f"  val={best_cp['val_loss']:.4f}, epoch={start_epoch}, step={global_step}")
    else:
        print("Warning: best.pt not found — VLM backbone will be HuggingFace weights.")

    # Step 2: overlay predictor + LoRA + optimizer state from the latest *real* step
    # checkpoint. Exclude step_00000.pt — it is always a blank initialization checkpoint
    # saved at the very start of each run and contains no real training state.
    ckpts = sorted([
        f for f in os.listdir(CHECKPOINT_DIR)
        if f.startswith("step_") and f.endswith(".pt") and f != "step_00000.pt"
    ])
    if ckpts:
        latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
        print(f"Overlaying predictor + LoRA + optimizer state from {latest}...")
        cp = torch.load(latest, map_location="cpu", weights_only=False)

        x_model.predictor.load_state_dict(cp["predictor"])
        with torch.no_grad():
            for name, param in y_encoder.named_parameters():
                if name in cp["y_lora"]:
                    param.copy_(cp["y_lora"][name])

        # Restore optimizer state for predictor + LoRA (prevents Adam cold-start spike).
        # VLM backbone optimizer state is not saved (too large); its momentum restarts.
        def _restore_opt(named_params, saved_key):
            saved = cp.get(saved_key, {})
            for name, param in named_params:
                if name in saved:
                    optimizer.state[param] = {
                        k: v.to(param.device) if isinstance(v, torch.Tensor) else v
                        for k, v in saved[name].items()
                    }

        _restore_opt(x_model.predictor.named_parameters(), "predictor_opt_state")
        _restore_opt(
            ((n, p) for n, p in y_encoder.named_parameters() if p.requires_grad),
            "lora_opt_state",
        )

        start_epoch = cp["epoch"]
        global_step = cp["global_step"]
        print(f"  Step checkpoint overlay: epoch={start_epoch}, "
              f"global_step={global_step}, val={cp['val_loss']:.4f}")
    else:
        print("  No step checkpoint beyond step_00000 — using best.pt state throughout.")

print(f"Training will resume from epoch={start_epoch}, global_step={global_step}")


Loading full checkpoint from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt...
  Restored Y-LoRA from best.pt
  val=0.0042, epoch=15, step=740
Overlaying predictor + LoRA + optimizer state from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/step_00740.pt...
  Step checkpoint overlay: epoch=15, global_step=740, val=0.0042
Training will resume from epoch=15, global_step=740


## Cell 6 — Live Y-Encoder target embedding

`compute_live_targets` is a gradient-enabled version of `embed_steps_batch()` from
`data/build_manifold.py`. The masking logic is identical — only the `torch.no_grad()`
wrapper is removed so LoRA adapter weights receive gradients.

For each step k, the function builds two VLM inputs (full and prefix), measures
token lengths, masks to only the new step tokens, and mean-pools the hidden states.

In [9]:
import torch.nn.functional as F


def compute_live_targets(images_batch, steps_batch, y_enc, y_proc, device, question_batch):
    # Gradient-enabled counterpart of embed_steps_batch() in data/build_manifold.py.
    # Step-token masking is identical; torch.no_grad() is removed for LoRA backprop.
    #
    # images_batch  : list[PIL.Image]  len B
    # steps_batch   : list[list[str]] [B, K]  cumulative steps from parse_k4_steps
    # question_batch: list[str]        len B
    # Returns       : Tensor [B, K, hidden_dim]

    B = len(images_batch)
    K = len(steps_batch[0])
    slot_embeddings = []

    for k in range(K):
        # Full: question + cumulative step k | Base: question + cumulative step k-1
        full_texts = [
            question_batch[b] + "\nAnswer: " + steps_batch[b][k]
            for b in range(B)
        ]
        base_texts = [
            question_batch[b] + "\nAnswer: " + (steps_batch[b][k - 1] if k > 0 else "")
            for b in range(B)
        ]

        msgs_full = [
            [{"role": "user", "content": [
                {"type": "image", "image": images_batch[b]},
                {"type": "text",  "text":  full_texts[b]},
            ]}]
            for b in range(B)
        ]
        msgs_base = [
            [{"role": "user", "content": [
                {"type": "image", "image": images_batch[b]},
                {"type": "text",  "text":  base_texts[b]},
            ]}]
            for b in range(B)
        ]

        prompts_full = [y_proc.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                        for m in msgs_full]
        prompts_base = [y_proc.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                        for m in msgs_base]

        # inputs_full goes to device — pixel_values needed for Y-encoder forward pass.
        inputs_full = y_proc(
            text=prompts_full, images=images_batch, return_tensors="pt", padding=True,
        ).to(device)

        # inputs_base stays on CPU — only attention_mask.sum() is used for masking;
        # pixel_values are never accessed, so there's no need to transfer them to GPU.
        inputs_base = y_proc(
            text=prompts_base, images=images_batch, return_tensors="pt", padding=True,
        )

        base_lens = inputs_base.attention_mask.sum(dim=1)   # CPU tensor, fine for indexing
        full_lens = inputs_full.attention_mask.sum(dim=1).cpu()

        # Identical masking logic as embed_steps_batch() in build_manifold.py
        target_mask = torch.zeros_like(inputs_full.attention_mask, dtype=torch.float32)
        pad_right = (y_proc.tokenizer.padding_side == "right")
        for b in range(B):
            b_len = int(base_lens[b].item())
            f_len = int(full_lens[b].item())
            if pad_right:
                target_mask[b, b_len:f_len] = 1.0
            else:
                seq = target_mask.shape[1]
                target_mask[b, seq - f_len + b_len : seq] = 1.0

        outputs = y_enc(
            input_ids=inputs_full.input_ids,
            attention_mask=inputs_full.attention_mask,
            pixel_values=inputs_full.get("pixel_values"),
            image_grid_thw=inputs_full.get("image_grid_thw"),
            mm_token_type_ids=inputs_full.get("mm_token_type_ids"),
            output_hidden_states=True,
            return_dict=True,
        )
        hidden = outputs.hidden_states[-1]  # [B, seq_len, dim]

        mask_exp = target_mask.unsqueeze(-1)
        sum_emb  = (hidden * mask_exp).sum(dim=1)
        sum_mask = target_mask.sum(dim=1, keepdim=True).clamp(min=1e-9)
        slot_embeddings.append(sum_emb / sum_mask)

    return torch.stack(slot_embeddings, dim=1)  # [B, K, dim]

## Cell 7 — Training loop

X-Encoder (VLM + predictor) and Y-Encoder (LoRA) are trained jointly.
Checkpoint saves predictor + LoRA adapters only (~200 MB); X-VLM backbone
is always recovered from HuggingFace.

In [12]:
import math, shutil
from collections import defaultdict
from transformers import get_cosine_schedule_with_warmup

train_loader = DataLoader(
    train_ds, batch_size=MICRO_BATCH_SIZE, shuffle=True,
    collate_fn=joint_collate, num_workers=4, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=MICRO_BATCH_SIZE, shuffle=False,
    collate_fn=joint_collate, num_workers=4, pin_memory=True,
)

history = defaultdict(list)

# Cosine schedule with linear warmup.
# last_epoch=global_step-1 makes this resume-aware: at a fresh start global_step=0
# so last_epoch=-1 (PyTorch default = start of training); after resuming it picks
# up at the correct point in the cosine curve without saving scheduler state.
# initial_lr must be set on param groups before constructing with last_epoch >= 0.
for _g in optimizer.param_groups:
    _g.setdefault("initial_lr", _g["lr"])
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps     = EPOCHS * steps_per_epoch
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
    last_epoch=global_step - 1,
)
print(f"Scheduler: cosine · {WARMUP_STEPS} warmup steps / {total_steps} total "
      f"({steps_per_epoch} steps/epoch)")


def _build_x_inputs(images, texts, device):
    msgs = [
        [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text":  txt},
        ]}]
        for img, txt in zip(images, texts)
    ]
    prompts = [
        x_model.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in msgs
    ]
    return x_model.processor(
        text=prompts, images=images, return_tensors="pt", padding=True,
    ).to(device)


def save_checkpoint(tag, epoch, step, val_loss, full=False):
    # Step checkpoints (full=False): predictor + y_lora + optimizer state — Phase 1 resume.
    # best.pt (full=True): adds model_state_dict (~8.5 GB) for Phase 2 compatibility
    #   (train_decoder_projection.py line 229: x_encoder.load_state_dict(state_dict["model_state_dict"])).
    # Optimizer state covers predictor + LoRA only; VLM backbone (~32 GB) excluded.
    def _opt_state(named_params):
        return {
            name: {k: v.cpu() if isinstance(v, torch.Tensor) else v
                   for k, v in optimizer.state[param].items()}
            for name, param in named_params
            if param in optimizer.state
        }

    lora_state = {k: v.cpu() for k, v in y_encoder.named_parameters() if v.requires_grad}
    payload = {
        "predictor":           x_model.predictor.state_dict(),
        "y_lora":              lora_state,
        "predictor_opt_state": _opt_state(x_model.predictor.named_parameters()),
        "lora_opt_state":      _opt_state(
            (n, p) for n, p in y_encoder.named_parameters() if p.requires_grad
        ),
        "epoch":       epoch,
        "global_step": step,
        "val_loss":    val_loss,
    }
    if full:
        payload["model_state_dict"] = {k: v.cpu() for k, v in x_model.state_dict().items()}
    path = os.path.join(CHECKPOINT_DIR, f"{tag}.pt")
    torch.save(payload, path)
    print(f"  Saved {tag}.pt ({os.path.getsize(path) / 1e6:.0f} MB)")
    return path


def run_validation():
    x_model.eval()
    y_encoder.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for images, texts, steps, questions in val_loader:
            inp = _build_x_inputs(images, texts, DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                pred = x_model(
                    input_ids=inp.input_ids,
                    attention_mask=inp.attention_mask,
                    pixel_values=inp.get("pixel_values"),
                    image_grid_thw=inp.get("image_grid_thw"),
                    mm_token_type_ids=inp.get("mm_token_type_ids"),
                )
                targ = compute_live_targets(
                    images, steps, y_encoder, processor_y, DEVICE, questions,
                )
                loss, _ = criterion(pred, targ.to(dtype=pred.dtype))
            total += loss.item()
            n += 1
    x_model.train()
    y_encoder.train()
    return total / max(n, 1)


print("Starting VL-JEPA joint training (X full + Y LoRA)...")
optimizer.zero_grad()
val_loss         = float("inf")
best_val_loss    = float("inf")
patience_counter = 0
early_stop       = False

x_model.train()
y_encoder.train()

if not RESUME:
    save_checkpoint(f"step_{0:05d}", epoch=start_epoch, step=0, val_loss=val_loss)

for epoch in range(start_epoch, EPOCHS):
    if early_stop:
        break

    for batch_idx, (images, texts, steps, questions) in enumerate(train_loader):

        inp = _build_x_inputs(images, texts, DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred = x_model(
                input_ids=inp.input_ids,
                attention_mask=inp.attention_mask,
                pixel_values=inp.get("pixel_values"),
                image_grid_thw=inp.get("image_grid_thw"),
                mm_token_type_ids=inp.get("mm_token_type_ids"),
            )
            targ = compute_live_targets(
                images, steps, y_encoder, processor_y, DEVICE, questions,
            )
            loss, metrics = criterion(pred, targ.to(dtype=pred.dtype))
            loss = loss / GRAD_ACCUM_STEPS

        loss.backward()

        is_update = (
            (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
            or (batch_idx + 1) == len(train_loader)
        )
        if is_update:
            all_params = (
                list(x_model.parameters())
                + [p for p in y_encoder.parameters() if p.requires_grad]
            )
            grad_norm = torch.nn.utils.clip_grad_norm_(all_params, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            train_loss = loss.item() * GRAD_ACCUM_STEPS
            cos_val    = metrics.get("loss/cosine_angular", 0.0)
            huber_val  = metrics.get("loss/huber_magnitude", 0.0)
            current_lr = scheduler.get_last_lr()[0]
            history["step"].append(global_step)
            history["train_loss"].append(train_loss)
            history["lr"].append(current_lr)

            print(
                f"Ep {epoch} | Step {global_step:4d} | "
                f"Loss {train_loss:.4f} | Cos {cos_val:.4f} | "
                f"Huber {huber_val:.4f} | GradNorm {float(grad_norm):.2f} | "
                f"LR {current_lr:.2e}"
            )

            if global_step % SAVE_EVERY_N_STEPS == 0:
                val_loss = run_validation()
                history["val_loss"].append((global_step, val_loss))
                print(f"  -> Val loss: {val_loss:.4f}")

                save_checkpoint(f"step_{global_step:05d}", epoch, global_step, val_loss)
                if val_loss < best_val_loss:
                    best_val_loss    = val_loss
                    patience_counter = 0
                    save_checkpoint("best", epoch, global_step, val_loss, full=True)
                    print(f"  New best: {best_val_loss:.4f}")
                else:
                    patience_counter += 1
                    print(f"  No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")
                    if patience_counter >= EARLY_STOP_PATIENCE:
                        print("  Early stopping triggered.")
                        early_stop = True
                        break

                # Keep latest 3 step checkpoints
                old = sorted(
                    f for f in os.listdir(CHECKPOINT_DIR)
                    if f.startswith("step_") and f.endswith(".pt")
                )
                for f in old[:-3]:
                    os.remove(os.path.join(CHECKPOINT_DIR, f))

msg = f"Early stop at step {global_step}." if early_stop else "Training complete."
print(msg)

Scheduler: cosine · 20 warmup steps / 4200 total (42 steps/epoch)
Starting VL-JEPA joint training (X full + Y LoRA)...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Ep 11 | Step  541 | Loss 0.0153 | Cos 0.0009 | Huber 0.0058 | GradNorm 0.10 | LR 9.62e-05
Ep 11 | Step  542 | Loss 0.1046 | Cos 0.0068 | Huber 0.0367 | GradNorm 3.16 | LR 9.62e-05
Ep 11 | Step  543 | Loss 0.1448 | Cos 0.0095 | Huber 0.0498 | GradNorm 1.90 | LR 9.62e-05
Ep 11 | Step  544 | Loss 0.0651 | Cos 0.0042 | Huber 0.0231 | GradNorm 0.58 | LR 9.62e-05
Ep 11 | Step  545 | Loss 0.0797 | Cos 0.0051 | Huber 0.0285 | GradNorm 1.22 | LR 9.62e-05
Ep 11 | Step  546 | Loss 0.0624 | Cos 0.0040 | Huber 0.0226 | GradNorm 0.96 | LR 9.61e-05
Ep 11 | Step  547 | Loss 0.0478 | Cos 0.0030 | Huber 0.0174 | GradNorm 0.55 | LR 9.61e-05
Ep 11 | Step  548 | Loss 0.0492 | Cos 0.0031 | Huber 0.0180 | GradNorm 0.51 | LR 9.61e-05
Ep 11 | Step  549 | Loss 0.0462 | Cos 0.0029 | Huber 0.0167 | GradNorm 0.60 | LR 9.61e-05
Ep 11 | Step  550 | Loss 0.0364 | Cos 0.0023 | Huber 0.0134 | GradNorm 0.57 | LR 9.61e-05
Ep 11 | Step  551 | Loss 0.0310 | Cos 0.0019 | Huber 0.0117 | GradNorm 0.39 | LR 9.61e-05
Ep 11 | St

KeyboardInterrupt: 

## Cell 8 — Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["step"], history["train_loss"], label="train", color="steelblue")
if history["val_loss"]:
    vsteps, vlosses = zip(*history["val_loss"])
    axes[0].plot(vsteps, vlosses, "o--", label="val", color="tomato")
axes[0].set_title("Huber-Cosine Loss — VL-JEPA Joint")
axes[0].set_xlabel("Global Step")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if len(history["train_loss"]) > 5:
    window = 5
    smoothed = [
        sum(history["train_loss"][max(0, i - window) : i + 1])
        / len(history["train_loss"][max(0, i - window) : i + 1])
        for i in range(len(history["train_loss"]))
    ]
    axes[1].plot(history["step"], smoothed, color="steelblue")
    axes[1].set_title("Train Loss (smoothed)")
    axes[1].set_xlabel("Global Step")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, "loss_curve.png"), dpi=120)
plt.show()

## Cell 9 — Cosine similarity sanity check

If joint training is working, mean cosine similarity should increase over training.
Compare per-step values against the frozen-Y baseline.

In [13]:
x_model.eval()
y_encoder.eval()

cos_per_step = [[] for _ in range(K_STEPS)]
N_CHECK = 20

with torch.no_grad():
    for i, (images, texts, steps, questions) in enumerate(val_loader):
        if i >= N_CHECK:
            break
        inp = _build_x_inputs(images, texts, DEVICE)
        pred = x_model(
            input_ids=inp.input_ids,
            attention_mask=inp.attention_mask,
            pixel_values=inp.get("pixel_values"),
            image_grid_thw=inp.get("image_grid_thw"),
            mm_token_type_ids=inp.get("mm_token_type_ids"),
        ).float()
        targ = compute_live_targets(
            images, steps, y_encoder, processor_y, DEVICE, questions,
        ).float()
        for k in range(K_STEPS):
            cos = F.cosine_similarity(pred[:, k, :], targ[:, k, :], dim=-1).mean().item()
            cos_per_step[k].append(cos)

print("Mean cosine similarity per thought slot (val, first 20 batches):")
for k in range(K_STEPS):
    mean_cos = sum(cos_per_step[k]) / len(cos_per_step[k])
    bar = "#" * int(max(0, mean_cos) * 40)
    print(f"  <thought_{k+1}>: {mean_cos:+.4f}  {bar}")

x_model.train()
y_encoder.train()

Mean cosine similarity per thought slot (val, first 20 batches):
  <thought_1>: +0.9998  #######################################
  <thought_2>: +0.9997  #######################################
  <thought_3>: +0.9997  #######################################
  <thought_4>: +0.9998  #######################################


PeftModel(
  (base_model): LoraModel(
    (model): Qwen3VLForConditionalGeneration(
      (model): Qwen3VLModel(
        (visual): Qwen3VLVisionModel(
          (patch_embed): Qwen3VLVisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3VLVisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3VLVisionAttention(
                (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                (proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3VLVisionMLP(
                (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
                (linear_fc

## Phase 2 — Decoder training (`train_decoder_projection.py`)

Trains the `prefix_projection` MLP that bridges X-Encoder latents into the frozen
`Qwen3-4B-Base` decoder. This is the unmodified existing script — called directly
via shell so no code is duplicated.

**What it does:**
- Loads X-Encoder from `best.pt` saved in Phase 1 and freezes it
- Trains only `YDecoderPrefix.prefix_projection` (+ optional unfrozen decoder layers
  per `train_decoder.unfreeze_layers` in `config.yaml`)
- Loss: cross-entropy against ground-truth answer strings
- Saves `decoder_best.pt` to `{CHECKPOINT_DIR}/decoder/`

**config.yaml defaults used** (`train_decoder` block):
`epochs=20`, `batch_size=32`, `lr=1e-5`, `unfreeze_layers=2`

**Note on images**: `train_decoder_projection.py` loads images from disk paths in the JSONL
(`GeoThought/playground/data/images/samples/sample_N.jpg` relative to `REPO_PATH`). Verify
the images are present before running Phase 2.

**Note on Y-Encoder**: Phase 2 does not use the Y-Encoder at all — it trains `prefix_projection`
using cross-entropy loss against ground-truth answer strings. The LoRA adapters trained in
Phase 1 are not needed here; they were only used to produce alignment targets during Phase 1.

In [10]:
# Free Phase 1 models from GPU before launching Phase 2 subprocess.
# Without this, the notebook kernel holds ~55 GB and the subprocess OOMs immediately.
import gc, torch

for _name in ["x_model", "y_encoder", "y_base", "optimizer", "criterion",
               "train_loader", "val_loader"]:
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()

_free  = torch.cuda.mem_get_info()[0] / 1e9
_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM free after cleanup: {_free:.1f} / {_total:.1f} GB")
print("Ready to launch Phase 2 (decoder training).")

VRAM free after cleanup: 101.3 / 102.0 GB
Ready to launch Phase 2 (decoder training).


In [11]:
import subprocess, tempfile, yaml

# Patch config to route decoder checkpoints to local CHECKPOINT_DIR
with open(os.path.join(REPO_PATH, "training/config.yaml")) as _f:
    _cfg = yaml.safe_load(_f)
_cfg.setdefault("train_decoder", {})["checkpoint_dir"] = CHECKPOINT_DIR
_tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False, dir=REPO_PATH)
yaml.dump(_cfg, _tmp)
_tmp.close()

_cfg_path = _tmp.name
_x_enc    = f"{CHECKPOINT_DIR}/best.pt"
_exp      = EXPERIMENT_NAME

print(f"Decoder will save to: {CHECKPOINT_DIR}/{EXPERIMENT_NAME}/decoder/")

import sys as _sys
_python = _sys.executable

_env = {**os.environ, "WANDB_MODE": "disabled", "PYTHONPATH": REPO_PATH}
_result = subprocess.run(
    [_python, "training/train_decoder_projection.py",
     "--config", _cfg_path,
     "--x_encoder_weights", _x_enc,
     "--experiment_name", _exp],
    cwd=REPO_PATH,
    env=_env,
)

os.unlink(_tmp.name)
print(f"Phase 2 exited with code {_result.returncode}")

Decoder will save to: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/

LatentEuclid Phase 4.5 (Y-Decoder Projection)
Executing with Configuration:
build_manifold:
  batch_size: 64
data:
  jsonl_path: data/geothoughts_arbitrary_cot.jsonl
experiment:
  name: v12_dynamic_sft
model:
  base_model_id: Qwen/Qwen3-VL-4B-Instruct
  decoder_base_model_id: Qwen/Qwen3-4B-Base
  k_steps: 4
  target_model_id: Qwen/Qwen3-VL-4B-Instruct
train_decoder:
  batch_size: 32
  checkpoint_dir: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder
  epochs: 20
  gradient_accumulation_steps: 1
  learning_rate: 1.0e-05
  max_grad_norm: 5.0
  unfreeze_layers: 2
  use_projection_mlp: true
  weight_decay: 0.01
  x_encoder_weights_override: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
train_end_to_end:
  batch_size: 1
  decoder_weights_override: /workspace/checkpoints/v2_huber_mean_pooled_contextual_thoughts/decoder/decoder_best.pt
 

Added 4 dynamically routed new thought tokens
Loading Base LatentEuclid Vision-Language Model (Qwen/Qwen3-VL-4B-Instruct)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 713/713 [00:01<00:00, 656.86it/s]


[cuda] Successfully loaded pre-aligned geometry from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
[cuda] Freezing X-Encoder (Method A)...
[cuda] Loading Phase 4 Y-Decoder Prefix...


Loading weights: 100%|██████████| 713/713 [00:00<00:00, 12982.53it/s]


Experimental: Unfreezing the first 2 layers of the base decoder!
Located 36 total transformer layers. Unfreezing first 2...
[cuda] Enabling Gradient Checkpointing on Y-Decoder for VRAM savings...
[cuda] Resumed Y-Decoder from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_epoch_1.pt (Starting at Epoch 2) | Historic Best Loss: 0.8179
Train split: 5618 | Val split: 625

=== DEBUG: Token Leakage Check ===
RAW PROMPT TOKENS: Given the figure, where points A, B, and C lie on a known circle, and in triangle ABC, angle ABC measures 70 degrees and angle ACB measures 30 degrees, if D is the midpoint of arc BAC and DB and DC are connected, what is the degree measure of angle DBC?<image>
Answer: <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|e

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


=== Epoch 2 Validation CE Loss: 0.7860 ===
[cuda] Saved checkpoint: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_epoch_2.pt
[cuda] New best validation loss 0.7860! Saved /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
=== Epoch 3 Validation CE Loss: 0.7726 ===
[cuda] Saved checkpoint: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_epoch_3.pt
[cuda] New best validation loss 0.7726! Saved /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
=== Epoch 4 Validation CE Loss: 0.7617 ===
[cuda] Saved checkpoint: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_epoch_4.pt
[cuda] New best validation loss 0.7617! Saved /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
[cuda] Auto-deleted ancient checkpoint decoder_epoch_0.pt.
==

## Phase 2 — End-to-end evaluation (`eval_e2e.py`)

Runs the full inference pipeline:
`image + question → X-Encoder → latent → prefix_projection → Qwen3-4B-Base → answer text`

Compares generated answers against ground-truth using exact match + numeric tolerance.
Compare the reported accuracy against **v4 SOTA baseline: 45.38%**.

In [17]:
import subprocess, sys as _sys

_dec    = f"{CHECKPOINT_DIR}/{EXPERIMENT_NAME}/decoder/decoder_best.pt"
_enc    = f"{CHECKPOINT_DIR}/best.pt"
_exp    = EXPERIMENT_NAME
_python = _sys.executable

print(f"X-Encoder: {_enc}")
print(f"Decoder:   {_dec}")

_env = {**os.environ, "PYTHONPATH": REPO_PATH}
_result = subprocess.run(
    [_python, "eval_e2e.py",
     "--config", "training/config.yaml",
     "--x_encoder_weights", _enc,
     "--decoder_weights", _dec,
     "--experiment_name_override", _exp],
    cwd=REPO_PATH,
    env=_env,
)
print(f"Eval exited with code {_result.returncode}")

X-Encoder: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
Decoder:   /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
Loading X-Encoder...
Added 4 dynamically routed new thought tokens


Loading Base LatentEuclid Vision-Language Model (Qwen/Qwen3-VL-4B-Instruct)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 713/713 [00:01<00:00, 647.58it/s]


Loaded X-Encoder weights from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
Loading Y-Decoder...


Loading weights: 100%|██████████| 713/713 [00:00<00:00, 13031.92it/s]


Experimental: Unfreezing the first 2 layers of the base decoder!
Located 36 total transformer layers. Unfreezing first 2...
Loaded Y-Decoder weights from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
Loading Validation Dataset...

================ END-TO-END EVALUATION (625 samples) ================



Evaluating Batches: 100%|██████████| 79/79 [00:42<00:00,  1.85it/s, Correct=289/625, Accuracy=46.24%]



+++ RESULTS +++
Evaluated 625 samples.
Correct: 289
Accuracy: 46.24%
Saved 625 evaluated samples to data/eval_vl_jepa_lora_v1.json
Eval exited with code 0


## Geometry3K Test Set Evaluation

Runs the same end-to-end inference pipeline on the official Geometry3K test set (601 samples).
Reports accuracy vs GeoThought validation set above.

In [18]:
import subprocess, sys as _sys

_dec    = f"{CHECKPOINT_DIR}/{EXPERIMENT_NAME}/decoder/decoder_best.pt"
_enc    = f"{CHECKPOINT_DIR}/best.pt"
_exp    = EXPERIMENT_NAME
_python = _sys.executable
_g3k_prompts = os.path.join(REPO_PATH, "GeoThought/evaluation_script/geometry3k_test_prompts.jsonl")
_g3k_img_root = os.path.join(REPO_PATH, "GeoThought/evaluation_script")

print(f"X-Encoder: {_enc}")
print(f"Decoder:   {_dec}")
print(f"Prompts:   {_g3k_prompts}")

_env = {**os.environ, "PYTHONPATH": REPO_PATH}
_result = subprocess.run(
    [_python, "eval_e2e.py",
     "--config", "training/config.yaml",
     "--x_encoder_weights", _enc,
     "--decoder_weights", _dec,
     "--experiment_name_override", _exp,
     "--dataset", "geometry3k",
     "--g3k_prompts", _g3k_prompts,
     "--g3k_img_root", _g3k_img_root],
    cwd=REPO_PATH,
    env=_env,
)
print(f"Geometry3K eval exited with code {_result.returncode}")

X-Encoder: /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
Decoder:   /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
Prompts:   /home/ec2-user/work/MMML-Project/GeoThought/evaluation_script/geometry3k_test_prompts.jsonl
Loading X-Encoder...


Added 4 dynamically routed new thought tokens
Loading Base LatentEuclid Vision-Language Model (Qwen/Qwen3-VL-4B-Instruct)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 713/713 [00:01<00:00, 647.77it/s]


Loaded X-Encoder weights from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/best.pt
Loading Y-Decoder...


Loading weights: 100%|██████████| 713/713 [00:00<00:00, 12705.48it/s]


Experimental: Unfreezing the first 2 layers of the base decoder!
Located 36 total transformer layers. Unfreezing first 2...
Loaded Y-Decoder weights from /home/ec2-user/work/MMML-Project/checkpoints/vl_jepa_lora/vl_jepa_lora_v1/decoder/decoder_best.pt
Loading Validation Dataset...

================ END-TO-END EVALUATION (601 samples) ================



Evaluating Batches: 100%|██████████| 76/76 [01:07<00:00,  1.13it/s, Correct=44/601, Accuracy=7.32%]



+++ RESULTS +++
Evaluated 601 samples.
Correct: 44
Accuracy: 7.32%
Saved 601 evaluated samples to data/eval_vl_jepa_lora_v1_g3k.json
Geometry3K eval exited with code 0
